# NYC Building Energy Benchmarking - Part A3: Exploratory Data Analysis (EDA)
## TY B.Tech Integrated Data Warehouse, OLAP and Data Mining Project
**Domain:** Building Energy and Sustainability Analytics (NYC Local Law 84)

### Objectives:
1. **Numerical Analysis:** Statistical distribution, skewness, kurtosis, boxplots
2. **Categorical Analysis:** Frequencies and proportions of borough, property types, performance bands
3. **Temporal Analysis:** Multi-year trend across 2022-2024
4. **Geographic Analysis:** Borough comparison of energy and GHG intensities
5. **Outcome Analysis:** Target distribution of high-energy buildings


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib'
CURATED_PATH = '../data/curated/nyc_building_energy_curated.csv'
PLOT_DIR = '../data/staging/eda_plots/'
os.makedirs(PLOT_DIR, exist_ok=True)

df = pd.read_csv(CURATED_PATH)
print(f"Loaded curated dataset: {len(df):,} records")


Loaded curated dataset: 7,370 records


### A. Numerical Distribution Analysis

In [2]:
measures = ['SiteEUI', 'SourceEUI', 'GrossFloorArea', 'GHGEmissions', 'EnergyStarScore']

stats = []
for m in measures:
    s = df[m].dropna()
    stats.append({
        'Measure': m,
        'Mean': round(float(s.mean()), 2),
        'Median': round(float(s.median()), 2),
        'Std Dev': round(float(s.std()), 2),
        'IQR': round(float(s.quantile(0.75) - s.quantile(0.25)), 2),
        'Skewness': round(float(s.skew()), 2),
        'Kurtosis': round(float(s.kurt()), 2)
    })

pd.DataFrame(stats)


### Visualizing Energy Intensity Distributions

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['SiteEUI'].dropna(), bins=40, color='teal', edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution of Site EUI (kBtu/sqft)')
axes[0].set_xlabel('Site EUI')
axes[0].set_ylabel('Building Frequency')
axes[0].grid(True, linestyle='--', alpha=0.5)

axes[1].boxplot(df['SiteEUI'].dropna(), vert=False, patch_artist=True, boxprops=dict(facecolor='lightblue'))
axes[1].set_title('Boxplot of Site EUI')
axes[1].set_xlabel('Site EUI')

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'site_eui_distribution.png'))
plt.show()


### B. Categorical Analysis

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['Borough'].value_counts().plot(kind='bar', ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_title('Building Benchmarks by Borough')
axes[0].set_ylabel('Building Count')
axes[0].tick_params(axis='x', rotation=45)

df['PropertyTypeGroup'].value_counts().plot(kind='bar', ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('Building Benchmarks by Property Type Group')
axes[1].set_ylabel('Building Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'categorical_distributions.png'))
plt.show()


### C. Temporal Multi-Year Analysis (2022–2024)

In [5]:
yearly = df.groupby('ReportingYear').agg({
    'PropertyID': 'count',
    'SiteEUI': 'mean',
    'GHGEmissions': 'sum'
}).rename(columns={'PropertyID': 'TotalBuildings', 'SiteEUI': 'AverageSiteEUI', 'GHGEmissions': 'TotalGHG'})

print("=== Multi-Year Benchmarking Trends ===")
print(yearly)


=== Multi-Year Benchmarking Trends ===
               TotalBuildings  AverageSiteEUI    TotalGHG
ReportingYear                                            
2022                     2452       84.241832  1365803.99
2023                     2454       82.955510  1353518.64
2024                     2464       81.534544  1339485.20


### D. Geographic / Borough-Level Intensity Analysis

In [6]:
borough_summary = df.groupby('Borough').agg({
    'SiteEUI': ['mean', 'median'],
    'GHGIntensity': 'mean',
    'GrossFloorArea': 'sum'
}).round(4)

print("=== Borough Performance Comparison ===")
print(borough_summary)


=== Borough Performance Comparison ===
               SiteEUI         GHGIntensity GrossFloorArea
                  mean  median         mean            sum
Borough                                                   
Bronx          83.9450  78.310       0.0054     60555389.0
Brooklyn       81.1635  76.960       0.0053    152811266.0
Manhattan      83.3058  77.680       0.0053    402899250.0
Queens         83.3686  77.055       0.0053    124142605.0
Staten Island  82.1657  78.565       0.0054     20678622.0


### E. Cross-Tabulation: Borough × Property Type Group Average SiteEUI

In [7]:
pivot_table = pd.pivot_table(
    df,
    values='SiteEUI',
    index='Borough',
    columns='PropertyTypeGroup',
    aggfunc='mean'
).round(2)

print("=== Mean Site EUI Cross-Tabulation ===")
pivot_table


=== Mean Site EUI Cross-Tabulation ===
